# DMPBridge Structure Evaluation

This notebook compares **manual/reference JSON blocks** with **Llama/AI detected JSON blocks** using three evaluation levels:

1. **Label evaluation** — how often the predicted label matches the reference.
2. **Narrative hierarchy evaluation** — whether the document label sequence is preserved, such as `document_title → section → content`.
3. **Boundary evaluation** — whether Llama places the split points between `section`, `subsection`, and `content` correctly.

This is designed for DMPBridge because the task is not only text generation or text similarity. The goal is to reconstruct the narrative structure of a DMP.


## Step 1 — Import libraries

In [37]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 300)


## Step 2 — Set project folders

Expected file names:

- `sample1_reference_blocks.json`
- `sample1_llama_blocks.json`

The code also handles names like:

- `sample1_reference__blocks.json`
- `sample1_llama_blocks.json`


In [38]:
# Find project root
cwd = Path.cwd()

if (cwd / "data").exists():
    project_root = cwd
elif (cwd.parent / "data").exists():
    project_root = cwd.parent
else:
    # Fallback for uploaded files in this environment
    project_root = Path("/mnt/data")

# Main folders
reference_dir = project_root / "data" / "reference_structure_blocks"
llama_dir = project_root / "data" / "llama3-3-70b_structured_blocks"

# If folders do not exist, use /mnt/data for uploaded sample files
if not reference_dir.exists():
    reference_dir = Path("/mnt/data")
if not llama_dir.exists():
    llama_dir = Path("/mnt/data")

output_dir = project_root / "data" / "llama3-3-70b_structure_evaluation_reports"
if project_root == Path("/mnt/data"):
    output_dir = Path("/mnt/data/llama3-3-70b_structure_evaluation_reports")

output_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Reference folder:", reference_dir)
print("Llama folder:", llama_dir)
print("Output folder:", output_dir)


Project root: c:\Users\Nahid\dmpbridge
Reference folder: c:\Users\Nahid\dmpbridge\data\reference_structure_blocks
Llama folder: c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structured_blocks
Output folder: c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports


## Step 3 — Find JSON files

In [39]:
reference_files = sorted(reference_dir.glob("*reference*blocks*.json"))
llama_files = sorted(llama_dir.glob("*llama*blocks*.json"))

print("Reference files:")
for f in reference_files:
    print(" -", f.name)

print("\nLlama files:")
for f in llama_files:
    print(" -", f.name)


Reference files:
 - sample10_reference_blocks.json
 - sample1_reference_blocks.json
 - sample2_reference_blocks.json
 - sample3_reference_blocks.json
 - sample4_reference_blocks.json
 - sample5_reference_blocks.json
 - sample6_reference_blocks.json
 - sample7_reference_blocks.json
 - sample8_reference_blocks.json
 - sample9_reference_blocks.json

Llama files:
 - sample10_llama_blocks.json
 - sample1_llama_blocks.json
 - sample2_llama_blocks.json
 - sample3_llama_blocks.json
 - sample4_llama_blocks.json
 - sample5_llama_blocks.json
 - sample6_llama_blocks.json
 - sample7_llama_blocks.json
 - sample8_llama_blocks.json
 - sample9_llama_blocks.json


## Step 4 — Helper functions

These functions:

- normalize text
- extract sample IDs
- load JSON blocks
- tokenize text
- calculate simple precision, recall, and F1


In [40]:
def normalize_label(label):
    """Normalize labels so small naming differences do not create false errors."""
    label = str(label or "").strip().lower()
    label = label.replace("-", "_").replace(" ", "_")

    mapping = {
        "title": "document_title",
        "documenttitle": "document_title",
        "document_title": "document_title",
        "section_title": "section",
        "section_heading": "section",
        "heading": "section",
        "sub_section": "subsection",
        "sub_section_heading": "subsection",
        "subsection_heading": "subsection",
        "paragraph": "content",
        "body": "content",
        "body_text": "content",
    }

    return mapping.get(label, label)


def normalize_text(text):
    """Clean text so spacing and punctuation differences have less impact."""
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = text.replace("“", '"').replace("”", '"').replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def tokenize(text):
    """
    Tokenize text for token-level evaluation.

    This keeps words, numbers, and punctuation as separate tokens.
    """
    text = normalize_text(text)
    return re.findall(r"\w+|[^\w\s]", text)


def token_set_similarity(a, b):
    """Word overlap similarity between two text blocks."""
    a_tokens = set(tokenize(a))
    b_tokens = set(tokenize(b))

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / len(a_tokens | b_tokens)


def sequence_similarity(a, b):
    """Character/order similarity between two text blocks."""
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()


def combined_similarity(a, b):
    """Final text similarity score: 60% sequence + 40% token overlap."""
    return (0.60 * sequence_similarity(a, b)) + (0.40 * token_set_similarity(a, b))


def sample_id_from_filename(path):
    """Extract sample ID from a file name."""
    name = path.stem.lower()

    remove_parts = [
        "_reference__blocks",
        "_reference_blocks",
        "_llama_blocks",
        "-reference-blocks",
        "-llama-blocks",
        "reference__blocks",
        "reference_blocks",
        "llama_blocks",
    ]

    for part in remove_parts:
        name = name.replace(part, "")

    name = name.strip("_- ")
    return name


def load_blocks(json_path):
    """Load blocks from JSON and keep index, label, and text."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Some JSON files may store blocks inside a dictionary
    if isinstance(data, dict):
        for key in ["blocks", "structured_blocks", "data", "items"]:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    blocks = []

    for i, block in enumerate(data, start=1):
        if not isinstance(block, dict):
            continue

        label = normalize_label(block.get("label", ""))
        text = str(block.get("text", "")).strip()

        if not text:
            continue

        blocks.append({
            "idx": i,
            "label": label,
            "text": text,
            "norm_text": normalize_text(text),
            "tokens": tokenize(text),
        })

    return blocks


def safe_divide(numerator, denominator):
    """Avoid division-by-zero errors."""
    return numerator / denominator if denominator else 0.0


def precision_recall_f1(tp, fp, fn):
    """Return precision, recall, and F1 from TP, FP, FN."""
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    f1 = safe_divide(2 * precision * recall, precision + recall)

    return precision, recall, f1


## Step 5 — Set evaluation rules

You can adjust these values.

- `SIMILARITY_THRESHOLD`: controls whether a Llama block is considered a text match for a reference block.
- `ORDER_TOLERANCE`: allows small block-index differences.
- `BOUNDARY_TOLERANCE_TOKENS`: allows boundary positions to be slightly off.


In [41]:
SIMILARITY_THRESHOLD = 0.75
ORDER_TOLERANCE = 1
BOUNDARY_TOLERANCE_TOKENS = 2

print("Similarity threshold:", SIMILARITY_THRESHOLD)
print("Order tolerance:", ORDER_TOLERANCE)
print("Boundary tolerance in tokens:", BOUNDARY_TOLERANCE_TOKENS)


Similarity threshold: 0.75
Order tolerance: 1
Boundary tolerance in tokens: 2


## Step 6 — Level 1A: Block-level matching

This is close to your original notebook.

It answers:

1. Did Llama find similar text?
2. Did Llama give the correct label?
3. Was the matched block in the correct order?

This is useful, but it can be strict when Llama merges part of content into a section heading.


In [42]:
def make_error_type(similar_text_found, label_correct, order_correct):
    """Simple explanation for each reference block."""
    if not similar_text_found:
        return "text_not_found"
    if similar_text_found and not label_correct:
        return "label_wrong"
    if similar_text_found and label_correct and not order_correct:
        return "order_wrong"
    return "correct"


def compare_blocks_by_similarity(sample_id, reference_blocks, llama_blocks,
                                 similarity_threshold=SIMILARITY_THRESHOLD,
                                 order_tolerance=ORDER_TOLERANCE):
    """Compare reference and Llama blocks using best text similarity matching."""

    used_llama_indices = set()
    rows = []

    for ref in reference_blocks:
        best = None
        best_score = -1

        for llama in llama_blocks:
            if llama["idx"] in used_llama_indices:
                continue

            score = combined_similarity(ref["text"], llama["text"])

            if score > best_score:
                best_score = score
                best = llama

        similar_text_found = best is not None and best_score >= similarity_threshold

        if similar_text_found:
            used_llama_indices.add(best["idx"])

            label_correct = ref["label"] == best["label"]
            order_difference = abs(ref["idx"] - best["idx"])
            order_correct = order_difference <= order_tolerance

            llama_idx = best["idx"]
            llama_label = best["label"]
            llama_text = best["text"]
        else:
            label_correct = False
            order_difference = None
            order_correct = False

            llama_idx = None
            llama_label = None
            llama_text = None

        rows.append({
            "sample_id": sample_id,
            "ref_idx": ref["idx"],
            "ref_label": ref["label"],
            "ref_text": ref["text"],
            "llama_idx": llama_idx,
            "llama_label": llama_label,
            "llama_text": llama_text,
            "similarity_score": round(best_score, 3) if best_score >= 0 else None,
            "similar_text_found": similar_text_found,
            "label_correct": label_correct,
            "order_difference": order_difference,
            "order_correct": order_correct,
            "error_type": make_error_type(similar_text_found, label_correct, order_correct),
        })

    extra_rows = []

    for llama in llama_blocks:
        if llama["idx"] not in used_llama_indices:
            extra_rows.append({
                "sample_id": sample_id,
                "llama_idx": llama["idx"],
                "llama_label": llama["label"],
                "llama_text": llama["text"],
                "issue": "extra_or_unmatched_llama_block",
            })

    return pd.DataFrame(rows), pd.DataFrame(extra_rows)


## Step 7 — Level 1B: Token-level label evaluation

This is stronger than simple block matching.

It answers:

> For each token in the document, did Llama assign the correct narrative label?

This catches boundary mistakes. For example, if Llama labels part of the content as `section`, token-level evaluation will penalize that.


In [43]:
def blocks_to_token_labels(blocks):
    """
    Convert blocks into one long token sequence and one label per token.

    Example:
    block 1 label = section, text = "1. Types of data."
    becomes tokens with label section.
    """
    tokens = []
    labels = []

    for block in blocks:
        block_tokens = block["tokens"]
        tokens.extend(block_tokens)
        labels.extend([block["label"]] * len(block_tokens))

    return tokens, labels


def align_token_labels_by_position(reference_blocks, llama_blocks):
    """
    Align reference and Llama token labels by token position.

    This assumes the two JSON files represent the same document in the same order.
    That is usually true in your DMPBridge evaluation files.
    """
    ref_tokens, ref_labels = blocks_to_token_labels(reference_blocks)
    llama_tokens, llama_labels = blocks_to_token_labels(llama_blocks)

    max_len = max(len(ref_tokens), len(llama_tokens))
    rows = []

    for i in range(max_len):
        ref_token = ref_tokens[i] if i < len(ref_tokens) else None
        llama_token = llama_tokens[i] if i < len(llama_tokens) else None

        ref_label = ref_labels[i] if i < len(ref_labels) else "missing_reference"
        llama_label = llama_labels[i] if i < len(llama_labels) else "missing_llama"

        rows.append({
            "token_position": i + 1,
            "ref_token": ref_token,
            "llama_token": llama_token,
            "token_match": ref_token == llama_token,
            "ref_label": ref_label,
            "llama_label": llama_label,
            "label_match": ref_label == llama_label,
        })

    return pd.DataFrame(rows)


def calculate_label_metrics(token_label_report, sample_id):
    """Calculate token-level precision, recall, and F1 for each label."""
    labels = sorted(
        set(token_label_report["ref_label"])
        | set(token_label_report["llama_label"])
    )

    rows = []

    for label in labels:
        if label in {"missing_reference", "missing_llama"}:
            continue

        tp = int(((token_label_report["ref_label"] == label) &
                  (token_label_report["llama_label"] == label)).sum())

        fp = int(((token_label_report["ref_label"] != label) &
                  (token_label_report["llama_label"] == label)).sum())

        fn = int(((token_label_report["ref_label"] == label) &
                  (token_label_report["llama_label"] != label)).sum())

        precision, recall, f1 = precision_recall_f1(tp, fp, fn)

        rows.append({
            "sample_id": sample_id,
            "label": label,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": round(precision, 3),
            "recall": round(recall, 3),
            "f1": round(f1, 3),
        })

    return pd.DataFrame(rows)


## Step 8 — Level 2: Narrative hierarchy evaluation

This compares the label sequence only.

Example:

```text
document_title → section → content → section → content
```

This tells you whether Llama preserved the overall DMP structure, even if a boundary is slightly wrong.


In [44]:
def get_label_sequence(blocks):
    """Return the ordered label sequence for a document."""
    return [block["label"] for block in blocks]


def compare_hierarchy(sample_id, reference_blocks, llama_blocks):
    """Compare the ordered narrative label sequence."""
    ref_sequence = get_label_sequence(reference_blocks)
    llama_sequence = get_label_sequence(llama_blocks)

    max_len = max(len(ref_sequence), len(llama_sequence))
    matched_positions = 0

    rows = []

    for i in range(max_len):
        ref_label = ref_sequence[i] if i < len(ref_sequence) else "missing_reference"
        llama_label = llama_sequence[i] if i < len(llama_sequence) else "missing_llama"
        match = ref_label == llama_label

        if match:
            matched_positions += 1

        rows.append({
            "sample_id": sample_id,
            "position": i + 1,
            "ref_label": ref_label,
            "llama_label": llama_label,
            "match": match,
        })

    sequence_exact_match = ref_sequence == llama_sequence
    hierarchy_position_accuracy = safe_divide(matched_positions, max_len)

    summary = {
        "sample_id": sample_id,
        "reference_sequence": " → ".join(ref_sequence),
        "llama_sequence": " → ".join(llama_sequence),
        "reference_blocks": len(ref_sequence),
        "llama_blocks": len(llama_sequence),
        "sequence_exact_match": sequence_exact_match,
        "hierarchy_position_accuracy": round(hierarchy_position_accuracy, 3),
    }

    return summary, pd.DataFrame(rows)


## Step 9 — Level 3: Boundary evaluation

This evaluates whether Llama places split points correctly.

A boundary is a transition such as:

```text
section → content
content → section
section → subsection
subsection → content
```

For each boundary, this code records:

- token position of the boundary
- label transition type

Then it calculates boundary precision, recall, and F1.


In [45]:
def get_boundaries(blocks):
    """
    Get boundary positions and transition types.

    Boundary position means the token index after each block ends.
    Example:
    token position 5 means the boundary occurs after token 5.
    """
    boundaries = []
    current_position = 0

    for i in range(len(blocks) - 1):
        current_position += len(blocks[i]["tokens"])

        boundaries.append({
            "boundary_id": i + 1,
            "position": current_position,
            "from_label": blocks[i]["label"],
            "to_label": blocks[i + 1]["label"],
            "transition": f'{blocks[i]["label"]} → {blocks[i + 1]["label"]}',
            "before_text": blocks[i]["text"],
            "after_text": blocks[i + 1]["text"],
        })

    return boundaries


def boundary_match(ref_boundary, llama_boundary, tolerance=BOUNDARY_TOLERANCE_TOKENS):
    """A boundary matches when transition type is same and token position is close enough."""
    same_transition = ref_boundary["transition"] == llama_boundary["transition"]
    close_position = abs(ref_boundary["position"] - llama_boundary["position"]) <= tolerance
    return same_transition and close_position


def compare_boundaries(sample_id, reference_blocks, llama_blocks,
                       tolerance=BOUNDARY_TOLERANCE_TOKENS):
    """Calculate boundary precision, recall, and F1."""
    ref_boundaries = get_boundaries(reference_blocks)
    llama_boundaries = get_boundaries(llama_blocks)

    used_llama = set()
    match_rows = []

    for ref_i, ref_boundary in enumerate(ref_boundaries):
        best_j = None
        best_distance = None

        for llama_j, llama_boundary in enumerate(llama_boundaries):
            if llama_j in used_llama:
                continue

            if ref_boundary["transition"] != llama_boundary["transition"]:
                continue

            distance = abs(ref_boundary["position"] - llama_boundary["position"])

            if distance <= tolerance:
                if best_distance is None or distance < best_distance:
                    best_distance = distance
                    best_j = llama_j

        if best_j is not None:
            used_llama.add(best_j)
            llama_boundary = llama_boundaries[best_j]

            match_rows.append({
                "sample_id": sample_id,
                "ref_boundary_id": ref_boundary["boundary_id"],
                "llama_boundary_id": llama_boundary["boundary_id"],
                "ref_position": ref_boundary["position"],
                "llama_position": llama_boundary["position"],
                "position_difference": abs(ref_boundary["position"] - llama_boundary["position"]),
                "ref_transition": ref_boundary["transition"],
                "llama_transition": llama_boundary["transition"],
                "boundary_match": True,
                "ref_before_text": ref_boundary["before_text"],
                "ref_after_text": ref_boundary["after_text"],
                "llama_before_text": llama_boundary["before_text"],
                "llama_after_text": llama_boundary["after_text"],
            })
        else:
            match_rows.append({
                "sample_id": sample_id,
                "ref_boundary_id": ref_boundary["boundary_id"],
                "llama_boundary_id": None,
                "ref_position": ref_boundary["position"],
                "llama_position": None,
                "position_difference": None,
                "ref_transition": ref_boundary["transition"],
                "llama_transition": None,
                "boundary_match": False,
                "ref_before_text": ref_boundary["before_text"],
                "ref_after_text": ref_boundary["after_text"],
                "llama_before_text": None,
                "llama_after_text": None,
            })

    extra_rows = []

    for llama_j, llama_boundary in enumerate(llama_boundaries):
        if llama_j not in used_llama:
            extra_rows.append({
                "sample_id": sample_id,
                "llama_boundary_id": llama_boundary["boundary_id"],
                "llama_position": llama_boundary["position"],
                "llama_transition": llama_boundary["transition"],
                "llama_before_text": llama_boundary["before_text"],
                "llama_after_text": llama_boundary["after_text"],
                "issue": "extra_or_unmatched_llama_boundary",
            })

    tp = int(sum(row["boundary_match"] for row in match_rows))
    fn = len(ref_boundaries) - tp
    fp = len(llama_boundaries) - tp

    precision, recall, f1 = precision_recall_f1(tp, fp, fn)

    summary = {
        "sample_id": sample_id,
        "reference_boundaries": len(ref_boundaries),
        "llama_boundaries": len(llama_boundaries),
        "matched_boundaries": tp,
        "extra_llama_boundaries": fp,
        "missed_reference_boundaries": fn,
        "boundary_precision": round(precision, 3),
        "boundary_recall": round(recall, 3),
        "boundary_f1": round(f1, 3),
    }

    return summary, pd.DataFrame(match_rows), pd.DataFrame(extra_rows)


## Step 10 — Match reference files with Llama files

In [46]:
reference_map = {sample_id_from_filename(f): f for f in reference_files}
llama_map = {sample_id_from_filename(f): f for f in llama_files}

common_sample_ids = sorted(set(reference_map) & set(llama_map))

print("Matched sample IDs:")
for sid in common_sample_ids:
    print(" -", sid)

missing_llama = sorted(set(reference_map) - set(llama_map))
missing_reference = sorted(set(llama_map) - set(reference_map))

if missing_llama:
    print("\nReference files without matching Llama file:", missing_llama)

if missing_reference:
    print("\nLlama files without matching reference file:", missing_reference)


Matched sample IDs:
 - sample1
 - sample10
 - sample2
 - sample3
 - sample4
 - sample5
 - sample6
 - sample7
 - sample8
 - sample9


## Step 11 — Run all evaluations

In [47]:
all_block_reports = []
all_extra_block_reports = []

all_token_label_reports = []
all_token_label_metrics = []

all_hierarchy_rows = []
all_hierarchy_position_reports = []

all_boundary_rows = []
all_boundary_match_reports = []
all_extra_boundary_reports = []

summary_rows = []

for sample_id in common_sample_ids:
    reference_blocks = load_blocks(reference_map[sample_id])
    llama_blocks = load_blocks(llama_map[sample_id])

    # Level 1A: block-level matching
    block_report, extra_block_report = compare_blocks_by_similarity(
        sample_id,
        reference_blocks,
        llama_blocks,
    )

    all_block_reports.append(block_report)
    all_extra_block_reports.append(extra_block_report)

    # Level 1B: token-level label evaluation
    token_label_report = align_token_labels_by_position(reference_blocks, llama_blocks)
    token_label_report["sample_id"] = sample_id
    token_label_metrics = calculate_label_metrics(token_label_report, sample_id)

    all_token_label_reports.append(token_label_report)
    all_token_label_metrics.append(token_label_metrics)

    token_label_accuracy = token_label_report["label_match"].mean() if not token_label_report.empty else 0

    # Level 2: hierarchy evaluation
    hierarchy_summary, hierarchy_position_report = compare_hierarchy(
        sample_id,
        reference_blocks,
        llama_blocks,
    )

    all_hierarchy_rows.append(hierarchy_summary)
    all_hierarchy_position_reports.append(hierarchy_position_report)

    # Level 3: boundary evaluation
    boundary_summary, boundary_match_report, extra_boundary_report = compare_boundaries(
        sample_id,
        reference_blocks,
        llama_blocks,
    )

    all_boundary_rows.append(boundary_summary)
    all_boundary_match_reports.append(boundary_match_report)
    all_extra_boundary_reports.append(extra_boundary_report)

    # Summary
    total_ref = len(reference_blocks)

    text_found = int(block_report["similar_text_found"].sum()) if not block_report.empty else 0
    block_label_correct = int(block_report["label_correct"].sum()) if not block_report.empty else 0
    order_correct = int(block_report["order_correct"].sum()) if not block_report.empty else 0

    summary_rows.append({
        "sample_id": sample_id,

        # File/block counts
        "reference_blocks": len(reference_blocks),
        "llama_blocks": len(llama_blocks),

        # Level 1A: block-level
        "block_text_match_rate": round(safe_divide(text_found, total_ref), 3),
        "block_label_accuracy": round(safe_divide(block_label_correct, total_ref), 3),
        "block_order_accuracy": round(safe_divide(order_correct, total_ref), 3),
        "extra_llama_blocks": len(extra_block_report),

        # Level 1B: token-level
        "token_label_accuracy": round(token_label_accuracy, 3),

        # Level 2: hierarchy
        "hierarchy_exact_match": hierarchy_summary["sequence_exact_match"],
        "hierarchy_position_accuracy": hierarchy_summary["hierarchy_position_accuracy"],

        # Level 3: boundary
        "boundary_precision": boundary_summary["boundary_precision"],
        "boundary_recall": boundary_summary["boundary_recall"],
        "boundary_f1": boundary_summary["boundary_f1"],
    })

block_level_report = pd.concat(all_block_reports, ignore_index=True) if all_block_reports else pd.DataFrame()
extra_llama_blocks_report = pd.concat(all_extra_block_reports, ignore_index=True) if all_extra_block_reports else pd.DataFrame()

token_label_report = pd.concat(all_token_label_reports, ignore_index=True) if all_token_label_reports else pd.DataFrame()
token_label_metrics = pd.concat(all_token_label_metrics, ignore_index=True) if all_token_label_metrics else pd.DataFrame()

hierarchy_report = pd.DataFrame(all_hierarchy_rows)
hierarchy_position_report = pd.concat(all_hierarchy_position_reports, ignore_index=True) if all_hierarchy_position_reports else pd.DataFrame()

boundary_report = pd.DataFrame(all_boundary_rows)
boundary_match_report = pd.concat(all_boundary_match_reports, ignore_index=True) if all_boundary_match_reports else pd.DataFrame()
extra_boundary_report = pd.concat(all_extra_boundary_reports, ignore_index=True) if all_extra_boundary_reports else pd.DataFrame()

summary_report = pd.DataFrame(summary_rows)

summary_report


C:\Users\Nahid\AppData\Local\Temp\ipykernel_22996\3007385214.py:104: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  boundary_match_report = pd.concat(all_boundary_match_reports, ignore_index=True) if all_boundary_match_reports else pd.DataFrame()


,sample_id,reference_blocks,llama_blocks,block_text_match_rate,block_label_accuracy,block_order_accuracy,extra_llama_blocks,token_label_accuracy,hierarchy_exact_match,hierarchy_position_accuracy,boundary_precision,boundary_recall,boundary_f1
0,sample1,28,28,1.000,1.000,1.000,0,0.921,True,1.000,0.185,0.185,0.185
1,sample10,13,13,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
2,sample2,12,38,0.750,0.583,0.250,29,0.883,False,0.132,0.027,0.091,0.042
3,sample3,14,10,0.571,0.214,0.214,2,0.675,False,0.214,0.000,0.000,0.000
4,sample4,2,2,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
5,sample5,13,28,0.692,0.692,0.077,19,0.859,False,0.321,0.000,0.000,0.000
6,sample6,11,11,0.273,0.273,0.273,8,0.829,True,1.000,0.500,0.500,0.500
7,sample7,2,2,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
8,sample8,13,13,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
9,sample9,11,12,1.000,0.818,1.000,1,0.985,False,0.083,0.455,0.500,0.476


## Step 12 — Main summary table

Recommended interpretation:

- `token_label_accuracy`: best simple score for segmentation quality.
- `hierarchy_exact_match`: whether the label sequence is preserved.
- `boundary_f1`: most important error-analysis score for section/content split mistakes.


In [48]:
summary_report

,sample_id,reference_blocks,llama_blocks,block_text_match_rate,block_label_accuracy,block_order_accuracy,extra_llama_blocks,token_label_accuracy,hierarchy_exact_match,hierarchy_position_accuracy,boundary_precision,boundary_recall,boundary_f1
0,sample1,28,28,1.000,1.000,1.000,0,0.921,True,1.000,0.185,0.185,0.185
1,sample10,13,13,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
2,sample2,12,38,0.750,0.583,0.250,29,0.883,False,0.132,0.027,0.091,0.042
3,sample3,14,10,0.571,0.214,0.214,2,0.675,False,0.214,0.000,0.000,0.000
4,sample4,2,2,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
5,sample5,13,28,0.692,0.692,0.077,19,0.859,False,0.321,0.000,0.000,0.000
6,sample6,11,11,0.273,0.273,0.273,8,0.829,True,1.000,0.500,0.500,0.500
7,sample7,2,2,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
8,sample8,13,13,1.000,1.000,1.000,0,1.000,True,1.000,1.000,1.000,1.000
9,sample9,11,12,1.000,0.818,1.000,1,0.985,False,0.083,0.455,0.500,0.476


## Step 13 — Token-level label metrics

In [49]:
token_label_metrics

,sample_id,label,tp,fp,fn,precision,recall,f1
0,sample1,content,979,42,46,0.959,0.955,0.957
1,sample1,document_title,5,0,0,1.000,1.000,1.000
2,sample1,section,36,20,20,0.643,0.643,0.643
3,sample1,subsection,92,30,30,0.754,0.754,0.754
4,sample10,content,1010,0,0,1.000,1.000,1.000
5,sample10,document_title,2,0,0,1.000,1.000,1.000
6,sample10,section,31,0,0,1.000,1.000,1.000
7,sample2,content,1996,145,138,0.932,0.935,0.934
8,sample2,document_title,7,0,0,1.000,1.000,1.000
9,sample2,section,9,32,15,0.220,0.375,0.277


## Step 14 — Label confusion matrix from token-level labels

This shows where Llama assigns the wrong label at the token level.

Example:

```text
reference = content
llama = section
```

This usually means Llama included content text inside a section heading.


In [50]:
if not token_label_report.empty:
    token_label_confusion = pd.crosstab(
        token_label_report["ref_label"],
        token_label_report["llama_label"],
        rownames=["Reference label"],
        colnames=["Llama label"],
    )
else:
    token_label_confusion = pd.DataFrame()

token_label_confusion


Llama label,content,document_title,missing_llama,section,subsection
Reference label,,,,,
content,9561,0,8,119,269
document_title,3,47,0,4,0
missing_reference,6,0,0,0,0
section,46,0,0,183,55
subsection,438,0,0,9,279


## Step 15 — Hierarchy report

In [51]:
hierarchy_report

,sample_id,reference_sequence,llama_sequence,reference_blocks,llama_blocks,sequence_exact_match,hierarchy_position_accuracy
0,sample1,document_title → section → subsection → content → subsection → content → subsection → content → section → content → section → content → ...,document_title → section → subsection → content → subsection → content → subsection → content → section → content → section → content → ...,28,28,True,1.000
1,sample10,document_title → section → content → section → content → section → content → section → content → section → content → section → content,document_title → section → content → section → content → section → content → section → content → section → content → section → content,13,13,True,1.000
2,sample2,document_title → section → subsection → content → section → subsection → content → section → subsection → content → section → content,document_title → section → content → subsection → content → subsection → content → subsection → content → section → subsection → content...,12,38,False,0.132
3,sample3,document_title → section → subsection → content → section → subsection → content → section → subsection → content → section → subsection...,content → section → content → subsection → content → subsection → content → subsection → content → subsection,14,10,False,0.214
4,sample4,document_title → content,document_title → content,2,2,True,1.000
5,sample5,document_title → section → content → section → content → section → content → section → content → section → content → section → content,document_title → section → content → subsection → content → subsection → content → section → content → subsection → content → subsection...,13,28,False,0.321
6,sample6,document_title → section → content → section → content → section → content → section → content → section → content,document_title → section → content → section → content → section → content → section → content → section → content,11,11,True,1.000
7,sample7,document_title → content,document_title → content,2,2,True,1.000
8,sample8,document_title → section → content → section → content → section → content → section → content → section → content → section → content,document_title → section → content → section → content → section → content → section → content → section → content → section → content,13,13,True,1.000
9,sample9,document_title → section → content → section → content → section → content → section → content → section → content,document_title → content → section → content → subsection → content → section → content → subsection → content → section → content,11,12,False,0.083


## Step 16 — Boundary report

In [52]:
boundary_report

,sample_id,reference_boundaries,llama_boundaries,matched_boundaries,extra_llama_boundaries,missed_reference_boundaries,boundary_precision,boundary_recall,boundary_f1
0,sample1,27,27,5,22,22,0.185,0.185,0.185
1,sample10,12,12,12,0,0,1.000,1.000,1.000
2,sample2,11,37,1,36,10,0.027,0.091,0.042
3,sample3,13,9,0,9,13,0.000,0.000,0.000
4,sample4,1,1,1,0,0,1.000,1.000,1.000
5,sample5,12,27,0,27,12,0.000,0.000,0.000
6,sample6,10,10,5,5,5,0.500,0.500,0.500
7,sample7,1,1,1,0,0,1.000,1.000,1.000
8,sample8,12,12,12,0,0,1.000,1.000,1.000
9,sample9,10,11,5,6,5,0.455,0.500,0.476


## Step 17 — Boundary match details

This table shows exactly which boundaries were correct or missed.

Important columns:

- `ref_transition`
- `ref_position`
- `llama_position`
- `position_difference`
- `boundary_match`


In [53]:
boundary_match_report

,sample_id,ref_boundary_id,llama_boundary_id,ref_position,llama_position,position_difference,ref_transition,llama_transition,boundary_match,ref_before_text,ref_after_text,llama_before_text,llama_after_text
0,sample1,1,1.0,5,5.0,0.0,document_title → section,document_title → section,True,DATA MANAGEMENT AND SHARING PLAN,Element 1: Data Type:,DATA MANAGEMENT AND SHARING PLAN,Element 1: Data Type:
1,sample1,2,2.0,11,11.0,0.0,section → subsection,section → subsection,True,Element 1: Data Type:,A. Types and amount of scientific data expected to be generated in the project:,Element 1: Data Type:,A. Types and amount of scientific data expected to be generated in the project:
2,sample1,3,3.0,27,27.0,0.0,subsection → content,subsection → content,True,A. Types and amount of scientific data expected to be generated in the project:,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly availab...",A. Types and amount of scientific data expected to be generated in the project:,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly availab..."
3,sample1,4,4.0,132,130.0,2.0,content → subsection,content → subsection,True,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly availab...","B. Scientific data that will be preserved and shared, and the rationale for doing so:","This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly availab...","B. Scientific data that will be preserved and shared, and the rationale for doing so:"
4,sample1,5,5.0,150,148.0,2.0,subsection → content,subsection → content,True,"B. Scientific data that will be preserved and shared, and the rationale for doing so:","As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha...","B. Scientific data that will be preserved and shared, and the rationale for doing so:","As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha..."
5,sample1,6,NaN,249,NaN,NaN,content → subsection,None,False,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha...","C. Metadata, other relevant data, and associated documentation:",None,None
6,sample1,7,NaN,261,NaN,NaN,subsection → content,None,False,"C. Metadata, other relevant data, and associated documentation:","In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...",None,None
7,sample1,8,NaN,306,NaN,NaN,content → section,None,False,"In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...","Element 2: Related Tools, Software and/or Code:",None,None
8,sample1,9,NaN,318,NaN,NaN,section → content,None,False,"Element 2: Related Tools, Software and/or Code:",Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,None,None
9,sample1,10,NaN,389,NaN,NaN,content → section,None,False,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,Element 3: Standards:,None,None


## Step 18 — Problem boundaries only

In [54]:
problem_boundaries = boundary_match_report[
    boundary_match_report["boundary_match"] == False
].copy()

problem_boundaries


,sample_id,ref_boundary_id,llama_boundary_id,ref_position,llama_position,position_difference,ref_transition,llama_transition,boundary_match,ref_before_text,ref_after_text,llama_before_text,llama_after_text
5,sample1,6,NaN,249,NaN,NaN,content → subsection,None,False,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha...","C. Metadata, other relevant data, and associated documentation:",None,None
6,sample1,7,NaN,261,NaN,NaN,subsection → content,None,False,"C. Metadata, other relevant data, and associated documentation:","In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...",None,None
7,sample1,8,NaN,306,NaN,NaN,content → section,None,False,"In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...","Element 2: Related Tools, Software and/or Code:",None,None
8,sample1,9,NaN,318,NaN,NaN,section → content,None,False,"Element 2: Related Tools, Software and/or Code:",Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,None,None
9,sample1,10,NaN,389,NaN,NaN,content → section,None,False,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,Element 3: Standards:,None,None
10,sample1,11,NaN,394,NaN,NaN,section → content,None,False,Element 3: Standards:,The following data will be created as a result of this project:Objective sedentary behavior metrics – no existing standards,None,None
11,sample1,12,NaN,415,NaN,NaN,content → section,None,False,The following data will be created as a result of this project:Objective sedentary behavior metrics – no existing standards,"Element 4: Data Preservation, Access, and Associated Timelines:",None,None
12,sample1,13,NaN,427,NaN,NaN,section → subsection,None,False,"Element 4: Data Preservation, Access, and Associated Timelines:",A. Repository where scientific data and metadata will be archived:,None,None
13,sample1,14,NaN,439,NaN,NaN,subsection → content,None,False,A. Repository where scientific data and metadata will be archived:,De-identified data will be uploaded to the University of California San Diego Library repository.,None,None
14,sample1,15,NaN,456,NaN,NaN,content → subsection,None,False,De-identified data will be uploaded to the University of California San Diego Library repository.,B. How scientific data will be findable and identifiable:,None,None


## Step 19 — Block-level report

In [55]:
display_cols = [
    "sample_id", "ref_idx", "ref_label", "llama_idx", "llama_label",
    "similarity_score", "similar_text_found", "label_correct",
    "order_difference", "order_correct", "error_type",
    "ref_text", "llama_text",
]

block_level_report[display_cols] if not block_level_report.empty else block_level_report


,sample_id,ref_idx,ref_label,llama_idx,llama_label,similarity_score,similar_text_found,label_correct,order_difference,order_correct,error_type,ref_text,llama_text
0,sample1,1,document_title,1.0,document_title,1.000,True,True,0.0,True,correct,DATA MANAGEMENT AND SHARING PLAN,DATA MANAGEMENT AND SHARING PLAN
1,sample1,2,section,2.0,section,1.000,True,True,0.0,True,correct,Element 1: Data Type:,Element 1: Data Type:
2,sample1,3,subsection,3.0,subsection,1.000,True,True,0.0,True,correct,A. Types and amount of scientific data expected to be generated in the project:,A. Types and amount of scientific data expected to be generated in the project:
3,sample1,4,content,4.0,content,0.985,True,True,0.0,True,correct,"This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly availab...","This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly availab..."
4,sample1,5,subsection,5.0,subsection,1.000,True,True,0.0,True,correct,"B. Scientific data that will be preserved and shared, and the rationale for doing so:","B. Scientific data that will be preserved and shared, and the rationale for doing so:"
5,sample1,6,content,6.0,content,0.976,True,True,0.0,True,correct,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha...","As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha..."
6,sample1,7,subsection,7.0,subsection,1.000,True,True,0.0,True,correct,"C. Metadata, other relevant data, and associated documentation:","C. Metadata, other relevant data, and associated documentation:"
7,sample1,8,content,8.0,content,1.000,True,True,0.0,True,correct,"In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...","In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host..."
8,sample1,9,section,9.0,section,1.000,True,True,0.0,True,correct,"Element 2: Related Tools, Software and/or Code:","Element 2: Related Tools, Software and/or Code:"
9,sample1,10,content,10.0,content,1.000,True,True,0.0,True,correct,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...


## Step 20 — Problem blocks only

In [56]:
problem_blocks = block_level_report[
    block_level_report["error_type"] != "correct"
].copy() if not block_level_report.empty else pd.DataFrame()

problem_blocks[display_cols] if not problem_blocks.empty else problem_blocks


,sample_id,ref_idx,ref_label,llama_idx,llama_label,similarity_score,similar_text_found,label_correct,order_difference,order_correct,error_type,ref_text,llama_text
43,sample2,3,subsection,3.0,content,1.000,True,False,0.0,True,label_wrong,Data management plans should describe whether and how data generated in the course of the proposed research will be shared and preserved...,Data management plans should describe whether and how data generated in the course of the proposed research will be shared and preserved...
44,sample2,4,content,NaN,None,0.581,False,False,NaN,False,text_not_found,"Roles & Responsibilities. For the proposed research, Director Samuel Stupp with help from the Executive Director of Research will take t...",None
45,sample2,5,section,15.0,section,1.000,True,True,10.0,False,order_wrong,2. Data used in publications,2. Data used in publications
46,sample2,6,subsection,NaN,None,0.261,False,False,NaN,False,text_not_found,"Data management plans should describe how data used in publications resulting from the proposed research will be made open, machine-read...",None
47,sample2,7,content,23.0,content,1.000,True,True,16.0,False,order_wrong,Research conducted within the Center will be published in appropriate scientific journals. The publisher may restrict the redistribution...,Research conducted within the Center will be published in appropriate scientific journals. The publisher may restrict the redistribution...
48,sample2,8,section,24.0,section,1.000,True,True,16.0,False,order_wrong,3. Data management resources,3. Data management resources
49,sample2,9,subsection,32.0,content,0.900,True,False,23.0,False,label_wrong,Data management plans should consult and reference available information about data management resources to be used in the course of the...,reference it in the DMP. Information about other Office of Science facilities can be found in the additional guidance from the sponspori...
50,sample2,10,content,NaN,None,0.481,False,False,NaN,False,text_not_found,DMPs should consult and reference available information about data management resources to be used in the course of the proposed researc...,None
51,sample2,11,section,37.0,section,1.000,True,True,26.0,False,order_wrong,"4. Confidentiality, security and rights","4. Confidentiality, security and rights"
52,sample2,12,content,38.0,content,1.000,True,True,26.0,False,order_wrong,"Data management plans must protect confidentiality, personal privacy, Personally Identifiable Information and U.S. national, homeland, a...","Data management plans must protect confidentiality, personal privacy, Personally Identifiable Information and U.S. national, homeland, a..."


## Step 21 — Extra/unmatched Llama blocks

In [57]:
extra_llama_blocks_report

,sample_id,llama_idx,llama_label,llama_text,issue
0,sample2,4,subsection,"Roles & Responsibilities. For the proposed research, Director Samuel Stupp with help from the",extra_or_unmatched_llama_block
1,sample2,5,content,Executive Director of Research will take the lead and responsibility for coordinating and ensuring data storage and access and communica...,extra_or_unmatched_llama_block
2,sample2,6,subsection,"Data Types and Sources. A brief, high-level description of the data to be generated or used through",extra_or_unmatched_llama_block
3,sample2,7,content,the course of the proposed research and which of these are considered Digital Research Data necessary to Validate the research findings....,extra_or_unmatched_llama_block
4,sample2,8,subsection,"Content and Format. A statement of plans for data and metadata content and format including, where",extra_or_unmatched_llama_block
5,sample2,9,content,"applicable, a description of documentation plans, annotation of relevant software, and the rationale for the selection of appropriate st...",extra_or_unmatched_llama_block
6,sample2,10,section,Data Sharing and Data Preservation. A description of the plans for data sharing and preservation.,extra_or_unmatched_llama_block
7,sample2,11,subsection,"This should include, where appropriate:",extra_or_unmatched_llama_block
8,sample2,12,content,the anticipated means for sharing and rationale for any restrictions on who may access the data and under what conditions;\n\na timeline...,extra_or_unmatched_llama_block
9,sample2,13,subsection,Rationale. A discussion of the rationale or justification for the proposed data management plan,extra_or_unmatched_llama_block


## Step 22 — Extra/unmatched Llama boundaries

In [58]:
extra_boundary_report

,sample_id,llama_boundary_id,llama_position,llama_transition,llama_before_text,llama_after_text,issue
0,sample1,6,245,content → subsection,"As this is a secondary data analysis project, we will only be able to publicly share in the UC San Diego Library Repository sitting beha...","C. Metadata, other relevant data, and associated documentation:",extra_or_unmatched_llama_boundary
1,sample1,7,257,subsection → content,"C. Metadata, other relevant data, and associated documentation:","In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...",extra_or_unmatched_llama_boundary
2,sample1,8,302,content → section,"In addition to the data described above, code and models will be included in the repository and in on our project’s GitHub website, host...","Element 2: Related Tools, Software and/or Code:",extra_or_unmatched_llama_boundary
3,sample1,9,314,section → content,"Element 2: Related Tools, Software and/or Code:",Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,extra_or_unmatched_llama_boundary
4,sample1,10,385,content → section,Data will be analyzed with custom code by our statistical and computer science team. ActiGraph data will be processed and analyzed using...,Element 3: Standards:,extra_or_unmatched_llama_boundary
5,sample1,11,390,section → content,Element 3: Standards:,The following data will be created as a result of this project:\n\nObjective sedentary behavior metrics – no existing standards,extra_or_unmatched_llama_boundary
6,sample1,12,411,content → section,The following data will be created as a result of this project:\n\nObjective sedentary behavior metrics – no existing standards,"Element 4: Data Preservation, Access, and Associated Timelines:",extra_or_unmatched_llama_boundary
7,sample1,13,423,section → subsection,"Element 4: Data Preservation, Access, and Associated Timelines:",A. Repository where scientific data and metadata will be archived:,extra_or_unmatched_llama_boundary
8,sample1,14,435,subsection → content,A. Repository where scientific data and metadata will be archived:,De-identified data will be uploaded to the University of California San Diego Library repository.,extra_or_unmatched_llama_boundary
9,sample1,15,452,content → subsection,De-identified data will be uploaded to the University of California San Diego Library repository.,B. How scientific data will be findable and identifiable:,extra_or_unmatched_llama_boundary


## Step 23 — Save all reports

In [59]:
summary_path = output_dir / "summary_report.csv"
block_path = output_dir / "block_level_report.csv"
problem_blocks_path = output_dir / "problem_blocks_report.csv"
extra_blocks_path = output_dir / "extra_llama_blocks_report.csv"

token_label_path = output_dir / "token_label_report.csv"
token_label_metrics_path = output_dir / "token_label_metrics.csv"
token_confusion_path = output_dir / "token_label_confusion_matrix.csv"

hierarchy_path = output_dir / "hierarchy_report.csv"
hierarchy_position_path = output_dir / "hierarchy_position_report.csv"

boundary_path = output_dir / "boundary_report.csv"
boundary_match_path = output_dir / "boundary_match_report.csv"
problem_boundaries_path = output_dir / "problem_boundaries_report.csv"
extra_boundaries_path = output_dir / "extra_llama_boundaries_report.csv"

summary_report.to_csv(summary_path, index=False)
block_level_report.to_csv(block_path, index=False)
problem_blocks.to_csv(problem_blocks_path, index=False)
extra_llama_blocks_report.to_csv(extra_blocks_path, index=False)

token_label_report.to_csv(token_label_path, index=False)
token_label_metrics.to_csv(token_label_metrics_path, index=False)
token_label_confusion.to_csv(token_confusion_path)

hierarchy_report.to_csv(hierarchy_path, index=False)
hierarchy_position_report.to_csv(hierarchy_position_path, index=False)

boundary_report.to_csv(boundary_path, index=False)
boundary_match_report.to_csv(boundary_match_path, index=False)
problem_boundaries.to_csv(problem_boundaries_path, index=False)
extra_boundary_report.to_csv(extra_boundaries_path, index=False)

print("Saved reports:")
for p in [
    summary_path,
    block_path,
    problem_blocks_path,
    extra_blocks_path,
    token_label_path,
    token_label_metrics_path,
    token_confusion_path,
    hierarchy_path,
    hierarchy_position_path,
    boundary_path,
    boundary_match_path,
    problem_boundaries_path,
    extra_boundaries_path,
]:
    print(" -", p)


Saved reports:
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\summary_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\block_level_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\problem_blocks_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\extra_llama_blocks_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\token_label_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\token_label_metrics.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\token_label_confusion_matrix.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\hierarchy_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\hierarchy_position_report.csv
 - c:\Users\Nahid\dmpbridge\data\llama3-3-70b_structure_evaluation_reports\bounda

## Step 24 — Recommended wording for your paper/poster

You can report the evaluation as:

> We evaluated DMPBridge structure extraction at three levels: block-level matching, token-level label assignment, and narrative boundary detection. Block-level matching measures whether each reference block was found with the correct label and order. Token-level label evaluation measures whether each token was assigned the correct narrative role (`document_title`, `section`, `subsection`, or `content`). Boundary evaluation measures whether transitions between structural elements were placed correctly, such as `section → content`. This is important because a model may preserve the overall label sequence while still shifting section/content boundaries.
